# CAT Saathi — train the intent model

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
No Drive, no accounts. At the end your browser downloads one zip — that is the model.


## 1. Get the code


In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader || echo 'NO GPU - switch runtime to T4'
!git clone -q https://github.com/krishagarwal314/sahayak-cat-operator-assistant.git /content/saathi 2>/dev/null || git -C /content/saathi pull -q
%cd /content/saathi/backend


## 2. Build the training data


In [ ]:
!python -m app.ai.intent.build_dataset


## 3. Train
About 5–10 minutes. If it stops, run this cell again — it resumes from the last checkpoint.


In [ ]:
!python -m app.ai.intent.train --out /content/intent-run --epochs 6


## 4. Try it


In [ ]:
import sys, json, torch
sys.path.insert(0, '/content/saathi/backend')
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from app.ai.intent.normalize import normalize
best = '/content/intent-run/best'
tok = AutoTokenizer.from_pretrained(best)
model = AutoModelForSequenceClassification.from_pretrained(best).eval()
labels = {int(k): v for k, v in json.load(open(best + '/labels.json')).items()}
def ask(text):
    with torch.no_grad():
        p = torch.softmax(model(**tok(normalize(text), return_tensors='pt')).logits[0], -1)
    top = p.topk(2)
    print(f'{text:42s} -> {labels[top.indices[0].item()]:20s} {top.values[0]:.0%}   (2nd {labels[top.indices[1].item()]} {top.values[1]:.0%})')
for q in ['how much fuel is left', 'is anything wrong with the machine', 'is it safe to work',
          'how long will this take', 'how do i start the machine', 'kitna diesel bacha hai',
          'मशीन में कोई खराबी है क्या', 'how long did i idle', 'how do i make tea']:
    ask(q)


## 5. Download the model


In [ ]:
from google.colab import files
files.download('/content/intent-run/intent-classifier.zip')


## 6. On Lightning
Drag `intent-classifier.zip` into the repo folder in Lightning's file browser, then in the terminal:
```bash
cd /teamspace/studios/this_studio/sahayak-cat-operator-assistant
unzip -o intent-classifier.zip -d backend/models/intent-classifier
CLASSIFIER_FIRST=1 bash scripts/demo.sh
```
